# Run-book 145-p12 — the cube is unchanged by the refactor

**What this proves:** a datacube built by `BASE` (a commit on `main`) and by this checkout from
the same real imagery is byte-for-byte equal, with equal metadata and equal sidecars (minus the
seconds). It is gate 4 for #145 P12a (Q7, the move of the builder's instrumentation) and for
P12b (B7, B14, and the cube-side B15 sites). It does **not** reach B16 (`rio_env` / `rio_open`
build the remote environment only for `abfss://` paths) or B17 (the download's `.part` cleanup):
P12b pins those with tests and says so.

**Claude does not run this** (spec 24). You run it cell by cell and paste back the **Summary**
cell's output, or a failing cell's message. Every check is a plain `assert` you can read: the
text above a check cell says what PASS means, and the `assert` says the same thing in code.

**Needs:** the Austria imagery and its catalog under the main checkout's
`tests/outputs/demo_e2e/imagery/` (gitignored), no network, no credentials. Disk: well under
1 GB. Time: an estimate of a few minutes per build, 4 builds in all (2 cases x 2 sides).

<!-- Rules for writing one (spec 24 amendment A1):
  - call fsd's public functions directly, keyword arguments spelled out;
  - small helpers only (~15 lines), defined just above their first use, with a docstring;
  - no environment variables, no try/except around a whole step, no result-file plumbing;
  - every step reads what it needs from disk + Settings, never from an earlier step's variables;
  - a plan cell (counts + GB, no transfer) before any step that moves more than ~1 GB;
  - a deliberate kill runs in a child process (os._exit would kill this kernel). -->

## Before you start

**Kernel — which Python runs the cells.** Use **`.venv`**: the Python at `fsd/.venv/bin/python`
in the **main** checkout (Python 3.11). It is the only environment with fsd's dependencies.
- Open VS Code on the **`fsd/` folder** (the main checkout) — even when this notebook lives in a
  worktree under `fsd/.claude/worktrees/<name>/`. Then **Select Kernel** (top right) →
  **Python Environments** → **`.venv`**.
- Not listed (e.g. VS Code was opened on the worktree folder, which has no `.venv` of its own)?
  **Select Another Kernel… → Python Environments → Enter interpreter path…** (wording varies by
  VS Code version) and give the main checkout's `fsd/.venv/bin/python`.
- The **Setup** cell below checks this and stops with a message if the kernel is wrong.

**Code — which `fsd` gets imported.** fsd is installed from the *main* checkout, so inside a
worktree a plain `import fsd` would quietly run `main`'s code, not the branch under test. The
Setup cell puts **this notebook's own** `src/` first on Python's import path and checks it worked.
(This is what the old run-books did with `export PYTHONPATH=src` — the same thing, invisibly.)

**Stop / resume.** Interrupt the kernel (■) to stop. Re-running a cell resumes: Step 2 skips a
build whose `.rev` stamp names the same commit. After a kernel restart, run **Setup** and
**Settings** again, then continue at any step (Step 3 and the Summary also need Step 1's
check cell, for `HEAD`) — every step reads what it needs from disk.

**When you are done.** Paste back what the **Summary** cell says to paste (plus any failing cell's
message). Then **Restart & Clear All Outputs** and save: this file is tracked in git, and a test
refuses saved outputs.

In [ ]:
# Setup -- which Python, and which fsd. Nothing to change here.
import sys
from pathlib import Path

# 1. Which Python? It must be fsd's own virtual environment, fsd/.venv.
VENV = Path(sys.prefix)
assert VENV.name == ".venv" and (VENV.parent / "pyproject.toml").exists(), (
    f"Wrong kernel: {sys.prefix} is not fsd's .venv. Pick the '.venv' kernel (see above).")
MAIN_CHECKOUT = VENV.parent              # the fsd checkout that owns the venv

# 2. Which fsd? The one next to this notebook: <checkout>/runbooks/<this> -> <checkout>/src.
#    Jupyter and VS Code start the kernel in the notebook's own folder.
CHECKOUT = Path.cwd().parent
assert Path.cwd().name == "runbooks" and (CHECKOUT / "src" / "fsd").is_dir(), (
    f"The kernel started in {Path.cwd()}, not in this notebook's runbooks/ folder.")
if str(CHECKOUT / "src") not in sys.path:
    sys.path.insert(0, str(CHECKOUT / "src"))
import fsd

assert Path(fsd.__file__).is_relative_to(CHECKOUT / "src"), (
    f"fsd was already imported from {fsd.__file__}. Restart the kernel, then run this cell first.")
print("Python :", sys.executable)
print("fsd    :", fsd.__file__)

## Settings

Every input this run-book uses. Change them here and nowhere else.

In [ ]:
# Settings
import datetime

BASE = "c6ba2c1"                 # the commit to compare against (P12b: P12a's merge commit)
OUT = MAIN_CHECKOUT / "tests" / "outputs" / "145_p12_cube_equality"   # gitignored
CATALOG = MAIN_CHECKOUT / "tests" / "outputs" / "demo_e2e" / "imagery" / "catalog.parquet"
ROI = CHECKOUT / "notebooks" / "shapefiles" / "s2grid=4772924.geojson"   # inside T33UWP, ~22 km2
STARTDATE = datetime.datetime(2018, 4, 1)
ENDDATE = datetime.datetime(2018, 10, 1)
BANDS = ["B04", "B08", "SCL"]    # the bands the imagery holds
MOSAIC_DAYS = 20
CASES = {
    # the serial logged loader and both sidecars, calendar windows (Q7)
    "calendar_logged": dict(mosaic_scheme="calendar", njobs_load_images=1,
                            write_timings=True, write_read_log=True),
    # images.load_images through mp.Pool and the acquisition windows (B7, B14)
    "acquisition_parallel": dict(mosaic_scheme="acquisition", njobs_load_images=2,
                                 write_timings=True),
}

## Step 1 — export the base tree

**What it does:** extracts `src/` of commit `BASE` into `OUT/base_src` (`git archive`), so a
child process can import that fsd instead of this checkout's. Skipped when the stamp file there
already names `BASE`.
**PASS if:** `OUT/base_src/src/fsd/__init__.py` exists, the stamp names `BASE`, `BASE` is an
ancestor of `HEAD`, and this checkout's `src/` has no uncommitted change (the branch side builds
`HEAD`'s code).
**Time / disk:** seconds; a few MB.

In [ ]:
# Step 1 -- export the base tree.
import shutil
import subprocess

BASE_SRC = OUT / "base_src"


def export_base_src():
    """Extract BASE's src/ into OUT/base_src, unless the stamp there already names BASE."""
    stamp = BASE_SRC / "BASE"
    if stamp.exists() and stamp.read_text().strip() == BASE:
        return
    shutil.rmtree(BASE_SRC, ignore_errors=True)
    BASE_SRC.mkdir(parents=True)
    archive = subprocess.run(["git", "archive", BASE, "src"], cwd=CHECKOUT, check=True,
                             capture_output=True).stdout
    subprocess.run(["tar", "-x"], cwd=BASE_SRC, input=archive, check=True)
    stamp.write_text(BASE + "\n")


export_base_src()

In [ ]:
# Step 1 -- check it.
def git(*args):
    """Run git in this checkout and return the CompletedProcess."""
    return subprocess.run(["git", *args], cwd=CHECKOUT, capture_output=True, text=True)


HEAD = git("rev-parse", "--short", "HEAD").stdout.strip()
assert (BASE_SRC / "src" / "fsd" / "__init__.py").exists(), "the base tree was not extracted"
assert (BASE_SRC / "BASE").read_text().strip() == BASE, "the stamp does not name BASE"
assert git("merge-base", "--is-ancestor", BASE, "HEAD").returncode == 0, (
    f"{BASE} is not an ancestor of HEAD ({HEAD})")
assert git("status", "--porcelain", "--", "src").stdout == "", (
    "src/ has uncommitted changes; commit them so the branch side builds HEAD")
print("BASE", BASE, "HEAD", HEAD)

## Step 2 — build every case with both sides

**What it does:** for each case in Settings, builds the same cube twice in a **child process**:
once importing the base tree (`OUT/<case>/base`) and once importing this checkout
(`OUT/<case>/branch`). Both go through the same helper, with the same catalog filter, ROI,
dates and bands, so the only difference is the fsd code. A folder is rebuilt unless its `.rev`
stamp names the commit it was built from.
**PASS if:** every output folder holds `datacube.npy` and `metadata.pickle.npy`.
**Time / disk:** an estimate of a few minutes per build; under 100 MB in all.

In [ ]:
# Step 2 -- build every case on both sides. BUILD_CODE is what each child process runs.
import json
import os

BUILD_CODE = """
import datetime, json, pathlib, sys
cfg = json.loads(sys.argv[1])
import fsd
assert pathlib.Path(fsd.__file__).is_relative_to(cfg["src_dir"]), fsd.__file__
import geopandas as gpd
from fsd.catalog.catalog import TileCatalog
from fsd.datacube import builder
start = datetime.datetime.fromisoformat(cfg["startdate"])
end = datetime.datetime.fromisoformat(cfg["enddate"])
shape = gpd.read_file(cfg["roi"])
subset = TileCatalog(cfg["catalog"]).filter(shape, start, end)
builder.build_datacube(
    catalog_subset=builder.flatten_catalog(subset), shape_gdf=shape, startdate=start,
    enddate=end, bands=cfg["bands"], mosaic_days=cfg["mosaic_days"],
    export_folderpath=cfg["out_dir"], if_missing_files="warn", **cfg["case"],
)
"""


def build(src_dir, out_dir, case):
    """Build one case in a child process whose fsd is imported from src_dir."""
    cfg = dict(src_dir=str(src_dir.resolve()), out_dir=str(out_dir), case=CASES[case],
               roi=str(ROI), catalog=str(CATALOG), startdate=STARTDATE.isoformat(),
               enddate=ENDDATE.isoformat(), bands=BANDS, mosaic_days=MOSAIC_DAYS)
    shutil.rmtree(out_dir, ignore_errors=True)
    subprocess.run([sys.executable, "-c", BUILD_CODE, json.dumps(cfg)], cwd=OUT, check=True,
                   env={**os.environ, "PYTHONPATH": str(src_dir)})


SIDES = {"base": (BASE_SRC / "src", BASE), "branch": (CHECKOUT / "src", HEAD)}
for case in CASES:
    for side, (src_dir, rev) in SIDES.items():
        out_dir, stamp = OUT / case / side, OUT / case / f"{side}.rev"
        if (out_dir / "datacube.npy").exists() and stamp.exists() \
                and stamp.read_text().strip() == rev:
            continue
        print("building", case, side, rev)
        build(src_dir, out_dir, case)
        stamp.write_text(rev + "\n")

In [ ]:
# Step 2 -- check it.
for case in CASES:
    for side in SIDES:
        for name in ("datacube.npy", "metadata.pickle.npy"):
            path = OUT / case / side / name
            assert path.exists(), f"missing {path}"
print("all", len(CASES) * len(SIDES), "builds are on disk")

## Step 3 — compare base and branch, per case

**What it does:** compares the two output folders of each case.
**PASS if,** for every case:
- both folders hold the same file names;
- `datacube.npy` has the same dtype and `np.array_equal` values;
- `metadata.pickle.npy` is equal (dicts by key, lists by item, arrays by `np.array_equal`,
  shapely geometries by `.equals`, anything else by `==`);
- `timings.json` is equal after dropping `total_seconds` and the *values* of `phase_seconds`
  (the keys and their order must match);
- `reads.jsonl` rows are equal after dropping `start`, `end` and `duration`.

A sidecar a case does not write is `None` in the Summary and is not compared.
**Time / disk:** seconds.

The first cell defines the helpers the Summary also uses: after a kernel restart, run it again.

In [ ]:
# Step 3 -- the comparison helpers.
import json

import numpy as np
from shapely.geometry.base import BaseGeometry

from fsd.storage import fs


def same(a, b):
    """Deep equality: dicts by key, lists by item, arrays by array_equal, geometries by .equals."""
    if isinstance(a, dict):
        return isinstance(b, dict) and a.keys() == b.keys() and all(same(a[k], b[k]) for k in a)
    if isinstance(a, (list, tuple)):
        return type(a) is type(b) and len(a) == len(b) and all(same(x, y) for x, y in zip(a, b))
    if isinstance(a, np.ndarray):
        return isinstance(b, np.ndarray) and a.dtype == b.dtype and np.array_equal(a, b)
    if isinstance(a, BaseGeometry):
        return isinstance(b, BaseGeometry) and a.equals(b)
    return bool(np.all(a == b))


def timings_without_seconds(path):
    """timings.json minus total_seconds, with phase_seconds reduced to its key order."""
    payload = json.loads(path.read_text())
    payload.pop("total_seconds")
    payload["phase_seconds"] = list(payload["phase_seconds"])
    return payload


def reads_without_times(path):
    """reads.jsonl rows minus start, end and duration."""
    rows = [json.loads(line) for line in path.read_text().splitlines()]
    return [{k: v for k, v in r.items() if k not in ("start", "end", "duration")} for r in rows]


def compare(case):
    """One bool per comparison for a case (None when neither side wrote that sidecar)."""
    base, branch = OUT / case / "base", OUT / case / "branch"
    cube = [np.load(d / "datacube.npy") for d in (base, branch)]
    meta = [fs.load_npy(str(d / "metadata.pickle.npy"), allow_pickle=True)[()]
            for d in (base, branch)]
    result = dict(
        shape=list(cube[0].shape),
        same_files=sorted(p.name for p in base.iterdir())
        == sorted(p.name for p in branch.iterdir()),
        cube_equal=cube[0].dtype == cube[1].dtype and np.array_equal(cube[0], cube[1]),
        metadata_equal=same(meta[0], meta[1]),
    )
    for key, name, reduce in [("timings_equal", "timings.json", timings_without_seconds),
                              ("reads_equal", "reads.jsonl", reads_without_times)]:
        both = [d / name for d in (base, branch)]
        result[key] = same(*(reduce(p) for p in both)) if all(p.exists() for p in both) else None
    return result

In [ ]:
# Step 3 -- check it.
for case in CASES:
    r = compare(case)
    assert r["same_files"], f"{case}: the two sides wrote different files"
    assert r["cube_equal"], f"{case}: datacube.npy differs (dtype or values)"
    assert r["metadata_equal"], f"{case}: metadata.pickle.npy differs"
    assert r["timings_equal"] in (True, None), f"{case}: timings.json differs"
    assert r["reads_equal"] in (True, None), f"{case}: reads.jsonl differs"
    print(case, "PASS", r["shape"])

## Summary — paste this output back

Re-reads everything from disk, so it works after a restart (run Setup, Settings and the Step 1
and Step 3 helper cells first).

In [ ]:
import json

summary = {"BASE": BASE, "HEAD": HEAD, **{case: compare(case) for case in CASES}}
print(json.dumps(summary, indent=2, default=str))